# Logistic Regression Implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq matplotlib numpy pandas scikit-learn seaborn


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import learning_curve

from pandas.api.types import CategoricalDtype

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)


### 03. Data Loading and Preprocessing


Looking at the dataset that will be processed, this is the **Car Evaluation Dataset** which contains data about car acceptability based on various attributes. The dataset was derived from a hierarchical decision model originally developed for demonstrating decision-making expert systems and is used for multi-class classification.

**Dataset characteristics:**
- **Total samples**: 1,728 observations
- **Features**: 6 categorical variables representing car characteristics
- **Target**: Multi-class classification (Car acceptability)
    - Class 0: unacc (unacceptable) - 1,210 samples (70.0%)
    - Class 1: acc (acceptable) - 384 samples (22.2%)
    - Class 2: good - 69 samples (4.0%)
    - Class 3: v-good (very good) - 65 samples (3.8%)

**Feature descriptions:**
- `buying`: Buying price (vhigh, high, med, low)
- `maint`: Maintenance price (vhigh, high, med, low)
- `doors`: Number of doors (2, 3, 4, 5more)
- `persons`: Capacity in terms of persons to carry (2, 4, more)
- `lug_boot`: Size of luggage boot (small, med, big)
- `safety`: Estimated safety of the car (low, med, high)

**Dataset context:**
This dataset was created from a hierarchical decision model that evaluates cars according to a structured concept hierarchy:
- **PRICE** (overall price): buying + maintenance costs
- **TECH** (technical characteristics): includes comfort and safety
- **COMFORT**: doors + persons + luggage boot capacity

**Key characteristics:**
- **Complete attribute space coverage**: The dataset contains all possible combinations of attribute values
- **No missing values**: All instances have complete information
- **Ordinal features**: All attributes have natural ordering (low → high, small → big, etc.)
- **Multi-class imbalance**: Strong class imbalance with 70% unacceptable cars

**Research applications:**
This dataset is particularly valuable for:
- **Multi-class classification**: Testing algorithms on imbalanced multi-class problems
- **Ordinal feature handling**: Understanding how algorithms process ordered categorical data
- **Decision tree analysis**: The hierarchical structure makes it ideal for tree-based methods
- **Feature importance studies**: Clear interpretable features for understanding model decisions
- **Constructive induction**: Testing structure discovery methods due to known concept hierarchy

**Real-world relevance:**
The dataset models practical car purchasing decisions where buyers consider multiple factors simultaneously. The hierarchical structure reflects how humans actually make complex decisions by breaking them into sub-problems (price vs. technical features vs. comfort).


In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/car_evaluation/car.data', sep = ',', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Checking the values and types of each column

for column in dataframe.columns:
    labels = dataframe[column].value_counts().index.tolist()
    counts = dataframe[column].value_counts().values.tolist()
    dtype = dataframe[column].dtype

    figure = plt.figure(figsize = (10, 8))
    plt.pie(counts, labels = labels, autopct = '%1.1f%%')
    plt.axis('equal')
    plt.legend(title = f"Counts: {counts}")
    plt.title(f"Distribution of {column} (dtype: {dtype})")
    plt.show()


In [ ]:
# Changing the data types of categorical columns

buying_type = CategoricalDtype(categories = ['vhigh', 'high', 'med', 'low'], ordered = True)
maint_type = CategoricalDtype(categories = ['vhigh', 'high', 'med', 'low'], ordered = True)
doors_type = CategoricalDtype(categories = ['2', '3', '4', '5more'], ordered = True)
persons_type = CategoricalDtype(categories = ['2', '4', 'more'], ordered = True)
lug_boot_type = CategoricalDtype(categories = ['small', 'med', 'big'], ordered = True)
safety_type = CategoricalDtype(categories = ['low', 'med', 'high'], ordered = True)

dataframe['buying'] = dataframe['buying'].astype(buying_type)
dataframe['maint'] = dataframe['maint'].astype(maint_type)
dataframe['doors'] = dataframe['doors'].astype(doors_type)
dataframe['persons'] = dataframe['persons'].astype(persons_type)
dataframe['lug_boot'] = dataframe['lug_boot'].astype(lug_boot_type)
dataframe['safety'] = dataframe['safety'].astype(safety_type)


In [ ]:
dataframe.head()


In [ ]:
# Converting categorical columns to numerical values

dataframe['buying'] = dataframe['buying'].cat.codes
dataframe['maint'] = dataframe['maint'].cat.codes
dataframe['doors'] = dataframe['doors'].cat.codes
dataframe['persons'] = dataframe['persons'].cat.codes
dataframe['lug_boot'] = dataframe['lug_boot'].cat.codes
dataframe['safety'] = dataframe['safety'].cat.codes
dataframe['class'] = dataframe['class'].astype('category').cat.codes

dataframe.head()


### 04. Data Visualization


In [ ]:
for label in dataframe:
    if label == 'class' or dataframe[label].dtype == 'object':
        continue

    plt.figure(figsize = (10, 6))

    for class_value in dataframe['class'].unique():
        subset = dataframe[dataframe['class'] == class_value]
        sns.kdeplot(subset[label], fill = True, alpha = 0.5, label = f'Class {class_value}')

    plt.title(f'Distribution of {label} by Class')
    plt.xlabel(label)
    plt.ylabel('Probability')
    plt.legend(title = 'Class')
    plt.grid()

    plt.show()


In [ ]:
# Checking the correlation matrix

plt.figure(figsize = (10, 8))
sns.heatmap(dataframe.corr(), annot = True, cmap = 'coolwarm', vmin = -1, vmax = 1)
plt.title('Correlation Matrix')
plt.show()


### 05. Dataset Splitting and Scaling


In [ ]:
# Shuffling the dataframe

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining the train, validation and test datasets sizes

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


In [ ]:
X_train, y_train = train_dataset.drop('class', axis = 1), train_dataset['class']
X_validation, y_validation = validation_dataset.drop('class', axis = 1), validation_dataset['class']
X_test, y_test = test_dataset.drop('class', axis = 1), test_dataset['class']


### 06. Logistic Regression Implementation and Evaluation


In [ ]:
# Logistic Regression implementation

logistic_regression_model = LogisticRegression(max_iter = 200)
logistic_regression_model.fit(X_train, y_train)

y_predictions_validation = logistic_regression_model.predict(X_validation)
y_predictions_test = logistic_regression_model.predict(X_test)


In [ ]:
# Classification reports

print('Validation Set Classification Report:')
print(classification_report(y_validation, y_predictions_validation))

print('Test Set Classification Report:')
print(classification_report(y_test, y_predictions_test))


In [ ]:
# Confusion matrix

confusion_matrix_validation = confusion_matrix(y_validation, y_predictions_validation)
confusion_matrix_test = confusion_matrix(y_test, y_predictions_test)

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_validation, annot = True, fmt = 'd', cmap = 'Blues', cbar = False)
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_test, annot = True, fmt = 'd', cmap = 'Greens', cbar = False)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Getting probability predictions for additional metrics

y_pred_proba_validation = logistic_regression_model.predict_proba(X_validation)
y_pred_proba_test = logistic_regression_model.predict_proba(X_test)


In [ ]:
# Basic accuracy metrics

accuracy_validation = accuracy_score(y_validation, y_predictions_validation)
accuracy_test = accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Accuracy: {accuracy_validation:.4f}')
print(f'Test Set Accuracy: {accuracy_test:.4f}')


In [ ]:
# Precision metrics

precision_validation = precision_score(y_validation, y_predictions_validation, average = 'weighted')
precision_test = precision_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Precision (weighted): {precision_validation:.4f}')
print(f'Test Set Precision (weighted): {precision_test:.4f}')

# Class-specific precision
precision_per_class_validation = precision_score(y_validation, y_predictions_validation, average = None)
precision_per_class_test = precision_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Precision:')
for i, prec in enumerate(precision_per_class_validation):
    print(f'\tClass {i}: {prec:.4f}')

print(f'\nTest Set - Class-specific Precision:')
for i, prec in enumerate(precision_per_class_test):
    print(f'\tClass {i}: {prec:.4f}')


In [ ]:
# Recall metrics

recall_validation = recall_score(y_validation, y_predictions_validation, average = 'weighted')
recall_test = recall_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Recall (weighted): {recall_validation:.4f}')
print(f'Test Set Recall (weighted): {recall_test:.4f}')

# Class-specific recall
recall_per_class_validation = recall_score(y_validation, y_predictions_validation, average = None)
recall_per_class_test = recall_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Recall:')
for i, rec in enumerate(recall_per_class_validation):
    print(f'\tClass {i}: {rec:.4f}')

print(f'\nTest Set - Class-specific Recall:')
for i, rec in enumerate(recall_per_class_test):
    print(f'\tClass {i}: {rec:.4f}')


In [ ]:
# F1-Score metrics

f1_validation = f1_score(y_validation, y_predictions_validation, average = 'weighted')
f1_test = f1_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set F1-Score (weighted): {f1_validation:.4f}')
print(f'Test Set F1-Score (weighted): {f1_test:.4f}')

# Class-specific F1-Score
f1_per_class_validation = f1_score(y_validation, y_predictions_validation, average = None)
f1_per_class_test = f1_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific F1-Score:')
for i, f1 in enumerate(f1_per_class_validation):
    print(f'\tClass {i}: {f1:.4f}')

print(f'\nTest Set - Class-specific F1-Score:')
for i, f1 in enumerate(f1_per_class_test):
    print(f'\tClass {i}: {f1:.4f}')


In [ ]:
# Balanced accuracy

balanced_acc_validation = balanced_accuracy_score(y_validation, y_predictions_validation)
balanced_acc_test = balanced_accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Balanced Accuracy: {balanced_acc_validation:.4f}')
print(f'Test Set Balanced Accuracy: {balanced_acc_test:.4f}')


In [ ]:
# Matthews Correlation Coefficient (MCC)

mcc_validation = matthews_corrcoef(y_validation, y_predictions_validation)
mcc_test = matthews_corrcoef(y_test, y_predictions_test)

print(f'Validation Set Matthews Correlation Coefficient: {mcc_validation:.4f}')
print(f'Test Set Matthews Correlation Coefficient: {mcc_test:.4f}')


In [ ]:
# Cohen's Kappa

kappa_validation = cohen_kappa_score(y_validation, y_predictions_validation)
kappa_test = cohen_kappa_score(y_test, y_predictions_test)

print(f'Validation Set Cohen\'s Kappa: {kappa_validation:.4f}')
print(f'Test Set Cohen\'s Kappa: {kappa_test:.4f}')


In [ ]:
# ROC AUC (One-vs-Rest for multi-class)

roc_auc_validation = roc_auc_score(y_validation, y_pred_proba_validation, multi_class = 'ovr')
roc_auc_test = roc_auc_score(y_test, y_pred_proba_test, multi_class = 'ovr')

print(f'Validation Set ROC AUC (One-vs-Rest): {roc_auc_validation:.4f}')
print(f'Test Set ROC AUC (One-vs-Rest): {roc_auc_test:.4f}')


In [ ]:
# Average Precision (PR AUC) for multi-class

avg_precision_validation = average_precision_score(y_validation, y_pred_proba_validation)
avg_precision_test = average_precision_score(y_test, y_pred_proba_test)

print(f'Validation Set Average Precision (PR AUC): {avg_precision_validation:.4f}')
print(f'Test Set Average Precision (PR AUC): {avg_precision_test:.4f}')


In [ ]:
# Log Loss

logloss_validation = log_loss(y_validation, y_pred_proba_validation)
logloss_test = log_loss(y_test, y_pred_proba_test)

print(f'Validation Set Log Loss: {logloss_validation:.4f}')
print(f'Test Set Log Loss: {logloss_test:.4f}')


In [ ]:
# ROC Curves for multi-class (One-vs-Rest) - Validation Set

from sklearn.preprocessing import label_binarize

# Binarize the output for multi-class ROC
y_val_bin = label_binarize(y_validation, classes = [0, 1, 2, 3])
n_classes = y_val_bin.shape[1]

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

plt.figure(figsize = (10, 8))

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_val_bin[:, i], y_pred_proba_validation[:, i])
    roc_auc[i] = roc_auc_score(y_val_bin[:, i], y_pred_proba_validation[:, i])
    plt.plot(fpr[i], tpr[i], lw = 2, label = f'Class {i} (AUC = {roc_auc[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw = 2, label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (One-vs-Rest) - Validation Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# ROC Curves for multi-class (One-vs-Rest) - Test Set

# Binarize the output for multi-class ROC
y_test_bin = label_binarize(y_test, classes = [0, 1, 2, 3])

# Compute ROC curve and ROC area for each class
fpr_test = dict()
tpr_test = dict()
roc_auc_test = dict()

plt.figure(figsize = (10, 8))

for i in range(n_classes):
    fpr_test[i], tpr_test[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba_test[:, i])
    roc_auc_test[i] = roc_auc_score(y_test_bin[:, i], y_pred_proba_test[:, i])
    plt.plot(fpr_test[i], tpr_test[i], lw = 2, label = f'Class {i} (AUC = {roc_auc_test[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw = 2, label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (One-vs-Rest) - Test Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curves for multi-class - Validation Set

precision_val = dict()
recall_val = dict()
avg_precision_val = dict()

plt.figure(figsize = (10, 8))

for i in range(n_classes):
    precision_val[i], recall_val[i], _ = precision_recall_curve(y_val_bin[:, i], y_pred_proba_validation[:, i])
    avg_precision_val[i] = average_precision_score(y_val_bin[:, i], y_pred_proba_validation[:, i])
    plt.plot(recall_val[i], precision_val[i], lw = 2, label = f'Class {i} (AP = {avg_precision_val[i]:.4f})')

# Add baseline for each class (proportion of positive samples)
for i in range(n_classes):
    baseline = np.mean(y_val_bin[:, i])
    plt.axhline(y = baseline, color = 'gray', linestyle = '--', alpha = 0.5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - Validation Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curves for multi-class - Test Set

precision_test = dict()
recall_test = dict()
avg_precision_test = dict()

plt.figure(figsize = (10, 8))

for i in range(n_classes):
    precision_test[i], recall_test[i], _ = precision_recall_curve(y_test_bin[:, i], y_pred_proba_test[:, i])
    avg_precision_test[i] = average_precision_score(y_test_bin[:, i], y_pred_proba_test[:, i])
    plt.plot(recall_test[i], precision_test[i], lw = 2, label = f'Class {i} (AP = {avg_precision_test[i]:.4f})')

# Add baseline for each class (proportion of positive samples)
for i in range(n_classes):
    baseline = np.mean(y_test_bin[:, i])
    plt.axhline(y = baseline, color = 'gray', linestyle = '--', alpha = 0.5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - Test Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Cross-validation with accuracy

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
cv_accuracy_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv = cv, scoring = 'accuracy')

print(f'Cross-validation Accuracy:')
print(f'\tMean: {cv_accuracy_scores.mean():.4f} (+/- {cv_accuracy_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_accuracy_scores]}')


In [ ]:
# Cross-validation with precision

cv_precision_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv = cv, scoring = 'precision_weighted')

print(f'Cross-validation Precision (weighted):')
print(f'\tMean: {cv_precision_scores.mean():.4f} (+/- {cv_precision_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_precision_scores]}')


In [ ]:
# Cross-validation with recall

cv_recall_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv = cv, scoring = 'recall_weighted')

print(f'Cross-validation Recall (weighted):')
print(f'\tMean: {cv_recall_scores.mean():.4f} (+/- {cv_recall_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_recall_scores]}')


In [ ]:
# Cross-validation with F1-Score

cv_f1_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv = cv, scoring = 'f1_weighted')

print(f'Cross-validation F1-Score (weighted):')
print(f'\tMean: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_f1_scores]}')


In [ ]:
# Cross-validation with ROC AUC

cv_roc_auc_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv = cv, scoring = 'roc_auc_ovr')

print(f'Cross-validation ROC AUC (One-vs-Rest):')
print(f'\tMean: {cv_roc_auc_scores.mean():.4f} (+/- {cv_roc_auc_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_roc_auc_scores]}')


In [ ]:
# Error analysis - Validation Set

misclassified_mask_validation = y_validation != y_predictions_validation
misclassified_indices_validation = np.where(misclassified_mask_validation)[0]

print(f'Validation Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_validation)} out of {len(y_validation)}')
print(f'Error rate: {len(misclassified_indices_validation)/len(y_validation)*100:.2f}%')

# Analyze misclassification by true class
class_names = ['unacc', 'acc', 'good', 'v-good']
for true_class in [0, 1, 2, 3]:
    true_class_mask = y_validation == true_class
    misclass_in_class = np.sum(misclassified_mask_validation & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    if total_in_class > 0:
        print(f'{class_names[true_class]} cars (Class {true_class}):')
        print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Error analysis - Test Set

misclassified_mask_test = y_test != y_predictions_test
misclassified_indices_test = np.where(misclassified_mask_test)[0]

print(f'Test Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_test)} out of {len(y_test)}')
print(f'Error rate: {len(misclassified_indices_test)/len(y_test)*100:.2f}%')

# Analyze misclassification by true class
for true_class in [0, 1, 2, 3]:
    true_class_mask = y_test == true_class
    misclass_in_class = np.sum(misclassified_mask_test & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    if total_in_class > 0:
        print(f'{class_names[true_class]} cars (Class {true_class}):')
        print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Confidence analysis

# Get the maximum probability for each prediction (confidence)
confidence_validation = np.max(y_pred_proba_validation, axis = 1)
confidence_test = np.max(y_pred_proba_test, axis = 1)

# Analyze confidence for correct vs incorrect predictions
correct_mask_validation = ~misclassified_mask_validation
correct_mask_test = ~misclassified_mask_test

print('Confidence Analysis:')
print(f'Validation Set:')
print(f'\tAverage confidence of correct predictions: {confidence_validation[correct_mask_validation].mean():.4f}')
print(f'\tAverage confidence of misclassified samples: {confidence_validation[misclassified_mask_validation].mean():.4f}')

print(f'Test Set:')
print(f'\tAverage confidence of correct predictions: {confidence_test[correct_mask_test].mean():.4f}')
print(f'\tAverage confidence of misclassified samples: {confidence_test[misclassified_mask_test].mean():.4f}')


In [ ]:
# Learning curve analysis

train_sizes = np.linspace(0.1, 1.0, 10)

train_sizes_abs, train_scores, val_scores = learning_curve(
    logistic_regression_model, X_train, y_train, train_sizes = train_sizes, cv = 5, 
    scoring = 'accuracy', random_state = 42, n_jobs = -1
)

train_mean = np.mean(train_scores, axis = 1)
train_std = np.std(train_scores, axis = 1)
val_mean = np.mean(val_scores, axis = 1)
val_std = np.std(val_scores, axis = 1)

plt.figure(figsize = (10, 6))
plt.plot(train_sizes_abs, train_mean, 'o-', color = 'blue', label = 'Training Accuracy')
plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha = 0.1, color = 'blue')

plt.plot(train_sizes_abs, val_mean, 'o-', color = 'red', label = 'Validation Accuracy')
plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha = 0.1, color = 'red')

plt.xlabel('Training Set Size')
plt.ylabel('Accuracy Score')
plt.title('Learning Curve - Logistic Regression')
plt.legend()
plt.grid(True)
plt.show()


### 07. Summary and Interpretation of Additional Metrics

The additional metrics provide deeper insights into your Logistic Regression model performance:

**Basic Classification Metrics:**
- **Balanced Accuracy**: Accounts for class imbalance better than regular accuracy
- **Matthews Correlation Coefficient (MCC)**: Measures correlation between observed and predicted classifications (-1 to +1, where +1 is perfect)
- **Cohen's Kappa**: Inter-rater reliability statistic that accounts for agreement by chance
- **Class-specific metrics**: Individual precision, recall, and F1 for each car acceptability class

**Probability-based Metrics:**
- **ROC-AUC (One-vs-Rest)**: Area under the ROC curve for multi-class problems, measures discriminative ability
- **Average Precision (PR-AUC)**: Area under Precision-Recall curve, better for imbalanced datasets like this one
- **Log Loss**: Quantifies the uncertainty of predictions based on probabilities

**Analysis Techniques:**
- **Cross-validation**: Provides robust estimates with confidence intervals
- **Error Analysis**: Identifies patterns in misclassified samples by car acceptability class
- **Learning Curve**: Shows if more data would improve performance
- **Confidence Analysis**: Examines prediction confidence for correct vs incorrect classifications

**ROC and PR Curves (Multi-class):**
- ROC curves show trade-off between sensitivity and specificity for each class vs rest
- PR curves are more informative for imbalanced datasets (especially for minority classes like "good" and "v-good")
- Higher AUC values indicate better performance for each class

**Multi-class Considerations:**
- **Class Imbalance**: The dataset has severe imbalance (70% unacceptable, only 3.8% very good)
- **One-vs-Rest Strategy**: Logistic regression treats each class against all others
- **Minority Class Performance**: Classes 2 (good) and 3 (v-good) are challenging due to few samples

These metrics help you:
1. **Assess model reliability** through cross-validation across multiple metrics
2. **Understand failure cases** through error analysis by car acceptability class
3. **Choose appropriate metrics** for imbalanced multi-class problems
4. **Compare models** objectively across multiple evaluation dimensions
5. **Evaluate probability calibration** through confidence analysis and log loss
6. **Identify class-specific strengths** and weaknesses in the model
